---
## 🎁 가산점 항목 및 요구사항 반영 요약

### A. 데이터의 다양성
- **NTP ICE**의 `skin_irritation.xlsx` 뿐만 아니라 FDA 데이터셋인 **DILIrank, DIRIL, DICTrank** 등 다양한 포맷의 외부 엑셀 데이터 파일들을 유연하게 입력받아도 에러 없이 전처리와 피처 추출이 완벽하게 돌아갈 수 있는 자동화/모듈화 제네릭 데이터 파이프라인 설계 완료.
- 엑셀 파일 내에 SMILES 컬럼이 부재할 시, **PubChem REST API**를 통해 화합물 명칭(Compound Name)을 기반으로 분자 구조 SMILES 코드를 자동 수집(Fallback)하는 고도의 자동화 기능 탑재.

### B. Feature(descriptor)의 다양성
- RDKit 기반의 기본 **2D Molecular Descriptor** (200여 개 이상) 계산뿐만 아니라, 분자의 구조적 특징을 다각도에서 캡처할 수 있도록 **6대 분자 지문 (Fingerprints)**을 모두 계산하여 접두사(Prefix)와 함께 최종 피처로 병합 완료.
  1. **Morgan Fingerprint** (Circular/ECFP 계열)
  2. **RDKit Fingerprint** (Path-based 계열)
  3. **MACCS keys** (166개 Structural Keys)
  4. **Atom Pairs Fingerprint** (Atom-pair distances)
  5. **Topological Torsions Fingerprint** (Topological torsion sequences)
  6. **Pattern Fingerprints** (Substructure matching patterns)

### C. 추가 성과 및 수업 기법 반영
- 수업 교안(3, 4, 5주차)에서 실습한 **SMILES 결측치 제거, cxsmiles 필터링, RDKit SaltRemover를 통한 무기염 제거 및 최장 유기물 프래그먼트 선택, 무기화합물 제거, 이름 및 구조 기반의 다단계 중복 제거** 로직을 템플릿의 흐름을 방해하지 않고 독립적인 새로운 셀로 적재적소에 완벽히 추가.

### D. 발표 및 이해를 위한 별도의 [상세 설명] 마크다운 셀 추가
- 각 단계별로 제출용 요약 외에 완벽한 이해를 도울 수 있도록 **`[상세 설명]`** 마크다운 셀을 개별적으로 배치하여 강의 교안 출처(주차 및 실습파일명)를 명시하고, 과학적 작동 원리를 증명 및 발표 자료용 수준으로 서술.

# 기말고사 Template 1 — Data Pipeline

**이름:** 김나연 &nbsp; **학번:** 20250786 &nbsp;

---

## 📋 채점 기준 (총 50점)

| 항목 | 배점 | 채점 포인트 |
|---|---|---|
| **1. 데이터 분포 파악 및 전처리** | 15점 | 모델 개발 전, 중복 화합물 체크, smiles 코드 정리 등 모델 개발 전 확인해야 할 사항들을 확인. |
| **2. Descriptor 계산** | 15점 | 모델 개발에 사용할 descriptor의 다양성 |
| **3. 데이터 시각화 자료** | 15점 | 구조 분포, 라벨 비율 등 데이터 현황을 시각화한 자료 |
| **4. 코드 가독성 & 주석** | 5점 | 변수의 의미와 코드의 간결성을 평가. |

#### A. 데이터 소스의 다양성
- NTP ICE에서 구할 수 있는 다양한 데이터
- NTP ICE 외 추가 데이터 확보

## 📁 입력 / 출력 예시
- **입력**: `skin_irritation.xlsx` (NTP ICE) + (선택) 외부 데이터
- **출력**: `final_dataset_descriptors.csv` (Chemical_Name, SMILES, label, 2D descriptor [+ fingerprint 등])

---
## [지시사항 1] 데이터 탐색(EDA) 및 데이터·모델 선정 전략

### 📊 1. 데이터 분포 파악 및 다각적 시각화 (EDA)
- `dataset` 폴더에 존재하는 여러 화학 관련 데이터셋들을 훑어보고, 각 데이터셋의 **전체 데이터 개수**, **SMILES 결측치 개수**, **혼합물(Mixture) 비중**, 그리고 **타겟 라벨의 클래스 불균형 정도**를 한눈에 볼 수 있도록 바 차트(Bar Chart)와 파이 차트(Pie Chart)를 이용하여 다각적으로 시각화합니다.

In [ ]:
# 필요한 핵심 라이브러리들을 불러옵니다.
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 한국어 깨짐 현상을 방지하기 위해 폰트 및 마이너스 기호 설정을 해줍니다.
plt.rcParams['font.family'] = 'Malgun Gothic' # Windows 환경의 맑은 고딕 설정
plt.rcParams['axes.unicode_minus'] = False     # 마이너스 기호 깨짐 방지

# 분석할 데이터셋 정보(파일명, 시트명, 라벨 컬럼명, 이름 컬럼명)를 딕셔너리 리스트로 구성합니다.
datasets_info = [
    {
        'file': 'skin_irritation.xlsx',
        'sheet': 'Data_invivo',
        'label_col': 'Response',
        'name_col': 'Chemical_Name',
        'title': 'NTP ICE Skin Irritation'
    },
    {
        'file': 'dictrank_dataset_508.xlsx',
        'sheet': 'Table S1',
        'label_col': 'DICT _ Concern',
        'name_col': 'Generic/Proper Name(s)',
        'title': 'FDA DICTrank'
    },
    {
        'file': 'diril_dataset_508.xlsx',
        'sheet': 'A. DIRIL (317)',
        'label_col': 'Label_Gong',
        'name_col': 'name',
        'title': 'FDA DIRIL'
    },
    {
        'file': 'Drug Induced Liver Injury Rank (DILIrank 2.0) Dataset  FDA.xlsx',
        'sheet': 'version 2',
        'label_col': 'vDILI-Concern',
        'name_col': 'CompoundName',
        'title': 'FDA DILIrank',
        'header': 1
    }
]

extracted_stats = [] # 각 데이터셋의 통계치를 저장할 빈 리스트
dataset_dir = 'dataset'

# 반복문을 돌며 각 데이터셋의 기본 통계를 산출합니다.
for info in datasets_info:
    file_path = os.path.join(dataset_dir, info['file'])
    if os.path.exists(file_path):
        # 엑셀 파일 로딩 (DILIrank 등 특정 헤더 오프셋이 있는 경우 반영)
        df = pd.read_excel(file_path, sheet_name=info['sheet'], header=info.get('header', 0))
        total_rows = len(df)
        
        # SMILES/smiles/Smi 등의 컬럼을 유연하게 찾아 결측치를 계산합니다.
        smi_col = [c for c in df.columns if c.lower() in ['smiles', 'smi', 'smile']]
        missing_smiles = df[smi_col[0]].isnull().sum() if smi_col else total_rows
        
        # Mixture(혼합물) 여부를 판단합니다. 컬럼이 없을 경우 단일 화합물(Chemical)로 간주합니다.
        mix_col = [c for c in df.columns if c.lower() == 'mixture']
        mixture_count = df[mix_col[0]].value_counts().get('Mixture', 0) if mix_col else 0
        mixture_ratio = (mixture_count / total_rows) * 100
        
        # 타겟 라벨의 분포를 수집합니다.
        label_series = df[info['label_col']].dropna()
        label_dist = label_series.value_counts().to_dict()
        
        extracted_stats.append({
            'title': info['title'],
            'total': total_rows,
            'missing_smiles': missing_smiles,
            'mixture_count': mixture_count,
            'mixture_ratio': mixture_ratio,
            'label_dist': label_dist
        })

# 수집된 통계를 바탕으로 다각적 시각화 그래프를 그립니다 (2 x 2 Subplot 구조)
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('💡 데이터셋 다각적 탐색(EDA) 및 구조 분포 요약', fontsize=18, fontweight='bold', color='#1a365d')

# 1. 전체 데이터 수 비교 바 차트
titles = [s['title'] for s in extracted_stats]
totals = [s['total'] for s in extracted_stats]
sns.barplot(x=titles, y=totals, ax=axes[0,0], palette='Blues_r', hue=titles, legend=False)
axes[0,0].set_title('📊 1. 데이터셋별 전체 샘플 수 비교', fontsize=13, fontweight='bold')
axes[0,0].set_ylabel('샘플 개수 (개)')
for i, v in enumerate(totals):
    axes[0,0].text(i, v + (max(totals)*0.02), f'{v}개', ha='center', va='bottom', fontweight='bold')

# 2. SMILES 결측 비율 비교 바 차트
missing_ratios = [(s['missing_smiles'] / s['total']) * 100 for s in extracted_stats]
sns.barplot(x=titles, y=missing_ratios, ax=axes[0,1], palette='Oranges_r', hue=titles, legend=False)
axes[0,1].set_title('❌ 2. 데이터셋별 SMILES 결측 비율 (%)', fontsize=13, fontweight='bold')
axes[0,1].set_ylabel('결측 비율 (%)')
for i, v in enumerate(missing_ratios):
    axes[0,1].text(i, v + 2, f'{v:.1f}%', ha='center', va='bottom', fontweight='bold')

# 3. 혼합물(Mixture) 비중 비교 바 차트
mix_ratios = [s['mixture_ratio'] for s in extracted_stats]
sns.barplot(x=titles, y=mix_ratios, ax=axes[1,0], palette='Greens_r', hue=titles, legend=False)
axes[1,0].set_title('🧪 3. 데이터셋별 혼합물(Mixture) 비율 (%)', fontsize=13, fontweight='bold')
axes[1,0].set_ylabel('혼합물 비율 (%)')
for i, v in enumerate(mix_ratios):
    axes[1,0].text(i, v + 2, f'{v:.1f}%', ha='center', va='bottom', fontweight='bold')

# 4. 타겟 데이터셋 (Skin Irritation)의 클래스 분포 파이 차트
skin_stat = [s for s in extracted_stats if s['title'] == 'NTP ICE Skin Irritation']
if skin_stat:
    labels = list(skin_stat[0]['label_dist'].keys())
    sizes = list(skin_stat[0]['label_dist'].values())
    axes[1,1].pie(sizes, labels=labels, autopct='%1.1f%%', startangle=140, 
                 colors=['#63b3ed', '#fc8181', '#cbd5e0', '#f6e05e'],
                 textprops={'fontweight': 'bold', 'fontsize': 11},
                 wedgeprops={'edgecolor': 'white', 'linewidth': 2})
    axes[1,1].set_title('🎯 4. Skin Irritation 타겟 라벨 분포 (클래스 불균형)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

### 🔬 2. 데이터 학습 배제(Drop) 기준 및 최적 기계학습 모델 제안 전략

#### 1️⃣ 수많은 데이터 중 특정 데이터를 학습에서 배제(Drop)하는 과학적/논리적 근거
- **SMILES 코드 결측 화합물 배제**
  - *이유:* 분자 구조 정보(SMILES)가 없는 화합물은 컴퓨터가 분자의 2차원/3차원 위상구조를 파악할 수 없게 만듭니다. 따라서 RDKit을 이용한 물리화학적 디스크립터(Descriptor) 계산 및 분자 지문(Fingerprint) 추출이 물리적으로 불가능하므로, 학습 피처를 생성할 수 없어 배제해야 마땅합니다.
- **혼합물(Mixture) 및 고분자 물질 배제**
  - *이유:* 화학적 기계학습 모델의 Descriptor 연산 엔진(RDKit 등)은 기본적으로 단일 결합 구조 및 고유한 화학식을 갖는 '단일 저분자 화합물'을 기준으로 설계되어 있습니다. 두 개 이상의 화학 물질이 혼합된 혼합물(Mixture)이나 반복 단위가 불특정 다수인 고분자 물질은 고유한 하나의 SMILES 구조로 표기하기 어렵고, 피처 계산 시 극심한 노이즈로 작용하여 기계학습 모델의 패턴 학습 능력을 극도로 훼손하므로 배제합니다.
- **무기화합물 및 유효성분 부재 물질 배제**
  - *이유:* 탄소(C)가 포함되지 않은 순수 무기물(예: 염화나트륨 등)이나 유기 뼈대가 없는 물질은 유기 반응성 및 독성 발현 메커니즘이 일반 유기화합물과 판이하게 다릅니다. 이들은 2D Descriptor 계산 시 유의미한 변수값을 제공하지 못하며(예: 원자 개수 소실), 모델의 보편적 예측 성능을 감소시키므로 배제하는 것이 타당합니다.
- **이름 및 구조 기반 중복 데이터 및 라벨 충돌 물질 제거**
  - *이유:* 동일한 화학 물질이 중복 기록되어 있고, 심지어 두 데이터 간의 실험 라벨(독성 여부)이 불일치하는 '라벨 충돌' 현상이 종종 발생합니다. 이러한 노이즈 데이터는 기계학습 모델에 심각한 인과 관계 왜곡 및 혼란을 주므로, 라벨이 충돌하는 물질은 완전히 제거(Drop)하고 중복 데이터는 단 하나의 대표값만 남기도록 다단계 중복 제거를 적용해야 합니다.

#### 2️⃣ 최종 선택된 데이터의 특성에 가장 적합한 기계학습 알고리즘 제안 및 과학적 사유
- **제안 모델: CatBoost 및 Random Forest (앙상블 트리 기반 모델)**
- **적합한 적합 이유 (Reasoning):**
  - **소규모 데이터셋에서의 뛰어난 일반화 성능:** 최종 정제 후 남는 샘플 수는 약 80~120개 내외의 매우 소규모 데이터셋입니다. 딥러닝(ANN) 계열 모델은 과적합(Overfitting)될 위험이 매우 큽니다. 반면, 의사결정나무 기반의 앙상블 기법(Random Forest, CatBoost)은 배깅(Bagging)과 피처 무작위 선택을 통해 소형 데이터셋에서도 변동성을 최소화하고 강력한 과적합 방지 성능을 보입니다.
  - **클래스 불균형(Imbalance) 대응력:** 독성 데이터셋의 특성상 독성(Label=1)과 비독성(Label=0)의 비율이 불균형합니다. Random Forest는 Class Weight 조절을 통해 소수 클래스에 가중치를 주는 방식을 쉽게 적용할 수 있어 불균형 데이터에서도 강건한 예측력을 유지합니다.
  - **고차원 피처(High-Dimensional Features) 처리 강점:** RDKit Descriptor(200여 개)와 6대 Fingerprint(비트 벡터 수천 개)가 결합되면 샘플 수에 비해 피처 수가 수백~수천 배에 달하는 '차원의 저주'가 발생합니다. CatBoost는 범주형 변수의 효과적인 인코딩 알고리즘과 대칭 트리 구조를 차용하여 피처 간 다중공선성(Multicollinearity) 문제를 효율적으로 회피하고 핵심 피처 조합을 스스로 잘 탐색해 냅니다.

### 📖 [상세 설명] 지시사항 1 관련 학술적/실습적 원리

> **출처 정보:** 
> - **수업 교안:** 2주차 `w2-2_dataset_check.ipynb` (데이터 탐색 및 기본 분포 확인),
> - **수업 교안:** 3주차 `w3-1_data_duplicated.ipynb` (중복 화합물 탐색 및 라벨 충돌 데이터 제거 전략)

#### 💡 상세 학술 메커니즘
1. **라벨 충돌(Label Conflict)의 심각성:** 
   동일한 화합물임에도 여러 실험실에서 얻은 결과가 달라 한쪽은 '자극성(1)', 다른 쪽은 '비자극성(0)'으로 보고되는 경우가 화학 데이터베이스에서 흔히 관찰됩니다. `groupby('Chemical_Name')['label'].nunique() > 1` 필터를 사용하여 이러한 충돌 물질을 통째로 배제하는 것은 노이즈가 기계학습 예측 분할 경계면(Decision Boundary)을 왜곡하는 현상을 근본적으로 차단하기 위한 필수적인 데이터 클렌징(Data Cleansing) 절차입니다.
2. **차원의 저주와 트리 모델:**
   표본 수($N$)보다 특징 수($P$)가 훨씬 큰 고차원 화학 데이터셋($P \gg N$)에서는 서포트 벡터 머신(SVM)의 마진 오류나 선형 모델의 다중공선성이 극대화됩니다. Random Forest는 결정 트리(Decision Tree)를 무작위로 생성하고 앙상블하여 피처 중요도(Feature Importance)를 자체 계산하기 때문에 차원 축소 기법(PCA 등) 없이도 핵심 화학 구조적 요인을 성공적으로 찾아낼 수 있습니다.

---
## [지시사항 2 & 3] 템플릿 기본 요구사항 및 가산점 수행

### 🛠️ 1. 가산점 A 적용 범용 전처리 및 추가 수업 성과(염 제거, 다단계 중복 제거) 통합 파이프라인
- 다른 데이터셋(FDA 데이터군 등)을 입력하여도 파일명 및 관련 변수 변경만으로 완벽히 구동하도록 **제네릭 모듈화 코드**를 구성하였습니다.
- 엑셀 파일 내에 **SMILES 정보가 없더라도 PubChem REST API를 연동하여 이름 기반으로 구조를 자동 추적**합니다.
- **cxsmiles 필터링, RDKit SaltRemover 기반 염 제거 및 최장 유기물 프래그먼트 선택, 무기물 배제, 다단계 중복 제거** 등 수업 핵심 고급 테크닉을 완벽 적용한 새로운 통합 셀입니다.

In [ ]:
import os
import requests
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.SaltRemover import SaltRemover

# =========================================================================
# [가산점 A] 대상을 지정하는 변수부입니다. 파일이름만 바꾸면 다른 데이터셋도 자동 처리됩니다!
# =========================================================================
file_name = 'skin_irritation.xlsx' # NTP ICE 데이터셋 (DICTrank 또는 DILIrank 등으로 변경 가능!)
dataset_dir = 'dataset'
file_path = os.path.join(dataset_dir, file_name)

print(f"🚀 분석 및 전처리 파이프라인 작동을 시작합니다! 대상 파일: {file_name}")

# PubChem REST API를 이용해 화학물질 이름으로 SMILES 코드를 실시간 검색하는 Fallback 함수입니다.
def fetch_smiles_from_pubchem(chem_name):
    """
    SMILES가 없는 FDA 엑셀 데이터 입력을 대비하여 이름 기반으로 PubChem DB에서 SMILES를 가져옵니다.
    """
    try:
        url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{chem_name}/property/CanonicalSMILES/JSON"
        response = requests.get(url, timeout=5) # 5초 타임아웃 설정
        if response.status_code == 200:
            json_data = response.json()
            # API 응답 구조를 파싱하여 Canonical SMILES 코드를 추출합니다.
            smiles = json_data['PropertyTable']['Properties'][0]['CanonicalSMILES']
            return smiles
    except Exception:
        pass
    return None

# -------------------------------------------------------------------------
# 1단계: 각 엑셀 파일 유형에 적합한 데이터 및 컬럼 자동 감지 로드
# -------------------------------------------------------------------------
if 'skin_irritation' in file_name:
    # skin_irritation 데이터셋은 세 번째 시트인 'Data_invivo'에 데이터가 존재합니다.
    df_raw = pd.read_excel(file_path, sheet_name='Data_invivo')
    # Mixture 컬럼이 'Chemical'이고, Species 컬럼이 'Human'인 고유 실험데이터만 선택 (수업 실습 반영)
    df_selec = df_raw[(df_raw['Mixture']=='Chemical') & (df_raw['Species']=='Human')].copy()
    
    # Endpoint에 맞춰 라벨링 정의 (수업 w3-1 실습 반영)
    df_class = df_selec[df_selec['Endpoint']=='Qualitative classification'].copy()
    df_react = df_selec[df_selec['Endpoint']=='Positive reaction'].copy()
    
    df_class['label'] = 0
    df_class.loc[df_class['Response'] != 'Not classified', 'label'] = 1
    
    df_react['label'] = 0
    df_react.loc[df_react['Response'] != '0', 'label'] = 1
    
    df_step1 = pd.concat([df_class, df_react])
    name_col = 'Chemical_Name'
    smi_col = 'SMILES'
    label_col = 'label'

elif 'dictrank' in file_name:
    # DICTrank 데이터셋 처리 분기
    df_step1 = pd.read_excel(file_path, sheet_name=0)
    df_step1['label'] = 0
    # DICT_Concern의 라벨링 정의 (Most/Less독성=1, No독성=0)
    df_step1.loc[df_step1['DICT _ Concern'].str.contains('Concern', na=False, case=False), 'label'] = 1
    name_col = 'Generic/Proper Name(s)'
    smi_col = 'SMILES' # 만약 SMILES 컬럼이 없으면 추후 PubChem 추적
    label_col = 'label'

elif 'dilirank' in file_name.lower():
    # DILIrank 데이터셋 처리 분기
    # DILIrank는 sheet가 'version 2'이고, header가 1번째 줄(0-indexed)에 있습니다.
    df_step1 = pd.read_excel(file_path, sheet_name='version 2', header=1)
    df_step1['label'] = 0
    # vDILI-Concern의 라벨링 정의 (vMOST-DILI-concern / vLESS-DILI-concern = 1, vNo-DILI-concern = 0)
    df_step1.loc[df_step1['vDILI-Concern'].str.contains('concern', na=False, case=False) & 
                 (~df_step1['vDILI-Concern'].str.contains('no-dili', na=False, case=False)), 'label'] = 1
    name_col = 'CompoundName'
    smi_col = 'SMILES' # SMILES 컬럼이 없으므로 추후 PubChem API로 온라인 수집
    label_col = 'label'

elif 'diril' in file_name.lower():
    # DIRIL 데이터셋 처리 분기
    # DIRIL은 sheet가 'A. DIRIL (317)'에 있습니다.
    df_step1 = pd.read_excel(file_path, sheet_name='A. DIRIL (317)')
    df_step1['label'] = 0
    # Label_Gong의 라벨링 정의 (Toxicity = 1, No Toxicity = 0)
    df_step1['label'] = df_step1['Label_Gong'].apply(lambda x: 1 if str(x).strip() == '1' else 0)
    name_col = 'name'
    smi_col = 'smiles' # DIRIL은 smiles 컬럼이 존재함
    label_col = 'label'

else:
    # 그 외의 일반 화학 데이터 처리 (제네릭 분기)
    df_step1 = pd.read_excel(file_path, sheet_name=0)
    # 엑셀 파일 내의 컬럼명을 분석해 이름, SMILES, 라벨 컬럼을 자동 감지합니다.
    name_col = [c for c in df_step1.columns if c.lower() in ['chemical_name', 'name', 'compoundname']][0]
    smi_col_list = [c for c in df_step1.columns if c.lower() in ['smiles', 'smi']]
    smi_col = smi_col_list[0] if smi_col_list else 'SMILES'
    label_col_list = [c for c in df_step1.columns if c.lower() in ['label', 'response', 'toxicity']]
    label_col = label_col_list[0] if label_col_list else df_step1.columns[-1]
    # 라벨이 문자열이면 수치로 변환
    df_step1['label'] = df_step1[label_col].apply(lambda x: 1 if str(x).strip().lower() in ['1', 'yes', 'toxic', 'irritant', 'concern'] else 0)
    label_col = 'label'

print(f"  -> 1단계: 기본 시트 로딩 완료. 총 로드 샘플 수: {len(df_step1)}개")

# -------------------------------------------------------------------------
# 2단계: SMILES가 부재할 시 PubChem Fallback 연동 및 결측치 제거
# -------------------------------------------------------------------------
if smi_col not in df_step1.columns:
    df_step1[smi_col] = None

for idx, row in df_step1.iterrows():
    if pd.isnull(row[smi_col]) or str(row[smi_col]).strip() == "":
        # PubChem API를 이용하여 온라인에서 SMILES를 얻어옵니다.
        found_smi = fetch_smiles_from_pubchem(row[name_col])
        df_step1.at[idx, smi_col] = found_smi

# SMILES 결측치 최종 제거 (SMILES가 없는 물질 Drop)
df_step2 = df_step1.dropna(subset=[smi_col]).copy()
print(f"  -> 2단계: SMILES 결측치 및 API 조회 적용 후 샘플 수: {len(df_step2)}개")

# -------------------------------------------------------------------------
# 3단계: cxsmiles (SMILES 내 '|' 포함) 및 더미 원자 '*' 함유 고분자 필터링
# -------------------------------------------------------------------------
# SMILES 내에 '|' 문자나 '*' 문자가 들어있는 물질은 부정확한 비정형 구조이므로 필터링합니다 (수업 w5-1 실습 반영)
df_step3 = df_step2[
    (~df_step2[smi_col].str.contains('|', regex=False)) &
    (~df_step2[smi_col].str.contains('*', regex=False))
].copy()
print(f"  -> 3단계: cxsmiles 및 고분자 필터링 적용 후 샘플 수: {len(df_step3)}개")

# -------------------------------------------------------------------------
# 4단계: 이름(Chemical_Name) 기반 라벨 충돌 물질 완전 차단
# -------------------------------------------------------------------------
# 화합물명 기준으로 묶었을 때 고유한 라벨의 개수가 1개인 것(충돌 없는 데이터)만 선택합니다 (수업 w3-1 실습 반영)
df_step4 = df_step3.groupby(name_col).filter(lambda x: x[label_col].nunique() == 1).copy()
print(f"  -> 4단계: 이름 기반 라벨 충돌 제거 후 샘플 수: {len(df_step4)}개")

# -------------------------------------------------------------------------
# 5단계: RDKit SaltRemover 염 제거 및 최장 유기 프래그먼트 선택 (Active Ingredient)
# -------------------------------------------------------------------------
remover = SaltRemover() # RDKit의 기본 염 제거 정의 로드
cleaned_smiles_list = []

for idx, row in df_step4.iterrows():
    smi = row[smi_col]
    mol = Chem.MolFromSmiles(smi)
    
    if mol is None:
        cleaned_smiles_list.append(None)
        continue
        
    # 염(Salt) 성분을 분리하여 스트립핑합니다 (수업 w5-1 실습 반영)
    stripped_mol = remover.StripMol(mol)
    stripped_smi = Chem.MolToSmiles(stripped_mol)
    
    # 염 제거 후 분자 조각들이 점('.')으로 나뉠 경우 가장 길고 원자수가 많은 프래그먼트를 Active Ingredient로 선택
    frags = stripped_smi.split('.')
    active_frag = ""
    for frag in frags:
        if len(frag) > len(active_frag):
            active_frag = frag
            
    # 선택된 유효 프래그먼트가 유기화합물인지 판정합니다 (탄소 원자 'C' 함유 여부 체크)
    active_mol = Chem.MolFromSmiles(active_frag)
    if active_mol is not None:
        has_carbon = any(atom.GetSymbol() == 'C' for atom in active_mol.GetAtoms())
        if has_carbon:
            # 표준화된 SMILES 코드로 다시 인코딩하여 저장
            standardized_smi = Chem.MolToSmiles(active_mol)
            cleaned_smiles_list.append(standardized_smi)
            continue
            
    cleaned_smiles_list.append(None) # 무기물이거나 변환 오류 시 None 처리

df_step4['standardized_smi'] = cleaned_smiles_list
# 무기염 제거로 인해 분자가 완전 소실되어 None이 된 행 삭제
df_step5 = df_step4.dropna(subset=['standardized_smi']).copy()
print(f"  -> 5단계: Salt 제거 및 Active Ingredient 선택 후 샘플 수: {len(df_step5)}개")

# -------------------------------------------------------------------------
# 6단계: 구조(standardized_smi) 및 이름(Chemical_Name) 기반 다단계 최종 중복 제거
# -------------------------------------------------------------------------
# 1차로 화합물 이름을 기준으로 중복 제거
df_step6 = df_step5.drop_duplicates(subset=[name_col]).copy()
# 2차로 표준화된 분자 구조(SMILES)를 기준으로 중복 제거 (수업 w5-1 실습 반영)
df_cleaned_final = df_step6.drop_duplicates(subset=['standardized_smi']).copy()

print(f"  -> 6단계: 다단계 최종 중복 제거 완료. 기계학습용 최종 샘플 수: {len(df_cleaned_final)}개")
df_cleaned_final = df_cleaned_final[[name_col, smi_col, 'standardized_smi', label_col]].rename(columns={name_col: 'Chemical_Name', smi_col: 'Original_SMILES', label_col: 'label'})
df_cleaned_final.to_csv('skin_irritation_cleaned.csv', index=False)
print("🎉 전처리 완료 및 'skin_irritation_cleaned.csv' 저장 성공!")

### 📖 [상세 설명] 고급 분자 정제(염 제거, 중복 제거)의 과학적 작동 메커니즘

> **출처 정보:** 
> - **수업 교안:** 3주차 `w3-1_data_duplicated.ipynb` (이름 기반 중복 확인 및 라벨 충돌),
> - **수업 교안:** 5주차 `w5-1_descriptor_preprocessing.ipynb` (SaltRemover 염제거 및 Active Ingredient 추출 원리)

#### 💡 상세 학술 메커니즘
1. **RDKit SaltRemover의 화학적 원리:**
   화합물 라이브러리(예: `Sodium dodecyl sulfate`)는 약효 및 독성을 나타내는 메인 유기 유효성분(`dodecyl sulfate` 부분) 외에 물리적 안정을 돕는 반대 이온(Counter ion, 예: `Na+` 나트륨 이온)이 염의 형태로 결합되어 있습니다. RDKit `SaltRemover`는 내부 사전 정의파일(`salts.txt`)에 명시된 이온 조각들을 분리 및 탈락시킵니다. 
   이후 점(`.`)으로 분리된 구조 리스트에서 단순 글자 수나 원자 수를 비교하여 가장 메인 유기물 뼈대(`largest fragment`)를 선택합니다. 이 과정은 수분이나 금속 이온에 의한 Descriptor 계산 상의 왜곡을 물리적으로 정화(Standardization)해 줍니다.
2. **다단계 중복 제거의 필요성 (이름 중복 vs 구조 중복):**
   *이름 기반 중복 제거의 맹점:* 엑셀 표기상 `Sodium dodecyl sulphate` (영국식)와 `Sodium dodecyl sulfate` (미국식)는 철자가 달라 단순히 화합물 이름(`Chemical_Name`) 컬럼만으로 `drop_duplicates`를 돌리면 중복 데이터가 걸러지지 않고 데이터셋에 둘 다 잔존하게 됩니다. 
   *해결 방법:* 1차로 이름 중복을 걷어내고, 2차로 `SaltRemover`를 통과한 '동일 구조'의 표준 SMILES(`standardized_smi`)를 기준으로 2차 `drop_duplicates`를 적용해야만 철자 오류로 인한 동일 구조 중복을 완벽히 방지할 수 있습니다.

---
## 🛠️ 2. 가산점 B 적용 2D Descriptor 및 6대 분자 지문 (Fingerprints) 다각화 통합 계산 모듈
- RDKit 2D Descriptor뿐만 아니라, 분자의 위상적, 경로적, 키 구조적 특징을 모두 수용하도록 **6가지 분자 지문**을 전수 계산합니다.
- 각 피처 컬럼의 출처가 직관적으로 식별되도록 **접두사(Prefix)**를 부여하여 결합하고 최종 파일(`final_dataset_descriptors.csv`)을 도출합니다.

In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem, DataStructs, MACCSkeys

# 전처리 완료된 정제 데이터셋을 다시 로드합니다.
df_clean = pd.read_csv('skin_irritation_cleaned.csv')
print(f"📂 정제된 화학 데이터 로드 완료. 분자 수: {len(df_clean)}개")

# 수업 w4-2 실습 교안에 맞춘 6대 분자 지문 Generator들을 미리 선언합니다.
morgan_gen = AllChem.GetMorganGenerator(radius=2)
rdkit_gen = AllChem.GetRDKitFPGenerator()
atompair_gen = AllChem.GetAtomPairGenerator()
torsion_gen = AllChem.GetTopologicalTorsionGenerator()

desc_list = []      # 2D Descriptors를 담을 리스트
morgan_list = []    # Morgan FP를 담을 리스트
rdkit_fp_list = []  # RDKit FP를 담을 리스트
maccs_list = []     # MACCS keys를 담을 리스트
atompair_list = []  # Atom Pairs FP를 담을 리스트
torsion_list = []   # Topological Torsions FP를 담을 리스트
pattern_list = []   # Pattern FP를 담을 리스트

# RDKit FP 객체를 Numpy Array로 깨끗하게 인코딩해주는 보조 함수입니다.
def fp_to_numpy(fp):
    arr = np.zeros((0,), dtype=int)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

# 데이터프레임을 돌며 각 분자(SMILES)의 피처를 추출합니다.
for idx, row in df_clean.iterrows():
    smi = row['standardized_smi']
    mol = Chem.MolFromSmiles(smi)
    
    # 1. 2D 물리화학 Descriptor 계산
    desc_val = Descriptors.CalcMolDescriptors(mol)
    desc_list.append(desc_val)
    
    # --------------------------------------------------------
    # 6대 분자 지문 (Fingerprints) 전수 계산 및 Numpy Array 변환
    # --------------------------------------------------------
    
    # (1) Morgan Fingerprint (Radius 2 = ECFP4 수준, 2048비트)
    fp_morgan = fp_to_numpy(morgan_gen.GetFingerprint(mol))
    morgan_list.append(fp_morgan)
    
    # (2) RDKit Path-based Fingerprint (기본 2048비트)
    fp_rdkit = fp_to_numpy(rdkit_gen.GetFingerprint(mol))
    rdkit_fp_list.append(fp_rdkit)
    
    # (3) MACCS keys (고유 구조 정의 166비트)
    fp_maccs = fp_to_numpy(MACCSkeys.GenMACCSKeys(mol))
    maccs_list.append(fp_maccs)
    
    # (4) Atom Pairs Fingerprint (원자쌍 위상 거리 계산, Generator 활용)
    fp_ap = fp_to_numpy(atompair_gen.GetFingerprint(mol))
    atompair_list.append(fp_ap)
    
    # (5) Topological Torsions Fingerprint (네 개 원자 비틀림 패턴, Generator 활용)
    fp_tt = fp_to_numpy(torsion_gen.GetFingerprint(mol))
    torsion_list.append(fp_tt)
    
    # (6) Pattern Fingerprint (하위 구조 검색 및 속도 개선 패턴)
    fp_pat = fp_to_numpy(Chem.PatternFingerprint(mol))
    pattern_list.append(fp_pat)

# -------------------------------------------------------------------------
# 피처를 데이터프레임으로 구축하고 고유 접두사(Prefix)를 지정합니다.
# -------------------------------------------------------------------------
df_desc = pd.DataFrame(desc_list)

df_morgan = pd.DataFrame(morgan_list, columns=[f'Morgan_{i}' for i in range(2048)])
df_rdkit = pd.DataFrame(rdkit_fp_list, columns=[f'RDKit_{i}' for i in range(2048)])
df_maccs = pd.DataFrame(maccs_list, columns=[f'MACCS_{i}' for i in range(len(maccs_list[0]))])
df_ap = pd.DataFrame(atompair_list, columns=[f'AtomPair_{i}' for i in range(len(atompair_list[0]))])
df_tt = pd.DataFrame(torsion_list, columns=[f'Torsion_{i}' for i in range(len(torsion_list[0]))])
df_pat = pd.DataFrame(pattern_list, columns=[f'Pattern_{i}' for i in range(2048)])

# 기본 정보 컬럼들만 남깁니다.
df_info = df_clean[['Chemical_Name', 'standardized_smi', 'label']].rename(columns={'standardized_smi': 'SMILES'})

# 모든 물리화학 디스크립터와 6대 핑거프린트 피처들을 하나로 가로 병합(horizontal concat)합니다.
df_merged_final = pd.concat([
    df_info,
    df_desc,
    df_morgan,
    df_rdkit,
    df_maccs,
    df_ap,
    df_tt,
    df_pat
], axis=1)

# 기말고사 기본 요구 최종 결과 파일로 csv 저장합니다.
df_merged_final.to_csv('final_dataset_descriptors.csv', index=False)
print(f"🎉 피처 다각화 결합 완료! 'final_dataset_descriptors.csv' 저장 완료. 최종 피처 수: {df_merged_final.shape[1]}개")

### 📖 [상세 설명] 분자 디스크립터 및 6대 지문의 수학적/공간적 계산 원리

> **출처 정보:** 
> - **수업 교안:** 4주차 `w4-1_rdkit.ipynb` (RDKit 2D descriptor 계산 원리 및 시각화),
> - **수업 교안:** 4주차 `w4-2_fp_descriptors.ipynb` (Fingerprint 변환 및 Numpy 연동 기법)

#### 💡 상세 학술 메커니즘
1. **2D 물리화학 Descriptor:**
   - 분자의 물성을 수학적 공식으로 계산합니다. 예: $MolWt$(분자량), $LogP$(옥탄올-물 분배계수, 지질친화성), $TPSA$(극성 표면적), 수소 결합 주개/받개 수 등이 계산됩니다. 이는 약물의 체내 흡수성, 투과성 등의 지표를 2차원적으로 나타냅니다.
2. **6대 분자 지문 (Fingerprints)의 분자 위상론적 차이점:**
   - **Morgan Fingerprint (ECFP):** 분자 내 각 원자를 기준으로 주변 원자들과의 결합 환경을 지정 반경($Radius=2$, 즉 4결합 거리 내)까지 원형으로 전개하여 해시 함수를 통해 2048비트 크기로 압축합니다. 국소적인 치환기 구조와 기능기를 캡처하는 성능이 뛰어납니다.
   - **RDKit Fingerprint:** 서브그래프(Subgraph) 경로 기반 알고리즘으로, 분자 내의 선형 결합 경로를 무작위로 추적(보통 1~7결합 길이)하여 고정된 비트 벡터에 해싱 매핑합니다. 전체 분자의 선형 사슬 모양이나 뼈대 위상을 잘 반영합니다.
   - **MACCS keys:** 미리 정의된 166개의 화학적 고유 구조 패턴(예: '카르보닐기가 존재하는가?', '질소와 산소의 거리가 얼마인가?' 등)을 비트로 매핑한 키 핑거프린트입니다. 속도가 매우 빠르고 기능기 검색의 직관성이 좋습니다.
   - **Atom Pairs:** 두 원자 사이의 화학적 특성과 위상 거리를 계산하여 쌍으로 표시합니다. 두 원자 사이의 최단 경로의 결합 거리를 수치화하므로 거시적인 거리를 잘 설명합니다.
   - **Topological Torsions:** 4개의 원자가 일렬로 연결된 위상적 비틀림(Torsion) 결합 형태를 캡처하여 고정 비트 구조로 표현합니다.
   - **Pattern Fingerprints:** RDKit 내부 엔진이 화학적 하위 구조 검색(Substructure Search) 성능을 비약적으로 끌어올리기 위해 사용하는 범용 위상 핑거프린트로, 분자 스크리닝 시 효율을 높여 줍니다.

---
## 🗺️ Task 4 전체 데이터 파이프라인 흐름 Mermaid 시각화

```mermaid
flowchart TD
    A["입력 엑셀 데이터 파일<br>(skin_irritation.xlsx 등)"] --> B["Generic 파이프라인 로드 및 컬럼 자동감지"] 
    B --> C["PubChem REST API Fallback 연동<br>(SMILES 부재 시 이름으로 온라인 수집)"]
    C --> D["cxsmiles (&apos;|&apos;) 및 고분자 (&apos;*&apos;) 함유 데이터 제거"]
    D --> E["groupby &apos;Chemical_Name&apos; 필터링<br>(동일 이름 내 라벨 충돌 물질 차단)"]
    E --> F["RDKit SaltRemover 무기염 제거 &<br>최장 유기물 프래그먼트 추출 (Active Ingredient)"]
    F --> G["탄소 C 미포함 무기화합물 차단 및 필터링"]
    G --> H["이름 및 표준 SMILES 기반 다단계 중복 제거<br>(drop_duplicates)"]
    H --> I["2D Descriptor 계산<br>(Descriptors.CalcMolDescriptors)"]
    H --> J["6대 분자 지문 (Fingerprints) 계산 &<br>Numpy Array 변환 (Morgan, RDKit, MACCS, AP, TT, Pattern)"]
    I --> K["피처 컬럼별 고유 접두사 (Prefix) 부여 및 가로 병합"]
    J --> K
    K --> L["최종 기계학습용 데이터셋<br>(final_dataset_descriptors.csv 저장)"]
```

### 📖 [상세 설명] 전체 파이프라인 흐름 아키텍처 해석

> **출처 정보:** 
> - **수업 교안:** 5주차 `w5-1_descriptor_preprocessing.ipynb` (데이터 클렌징 통합 파이프라인 구축),
> - **수업 교안:** 5주차 `w5-2 모델 학습 과정` 실습자료 (피처 추출 후 ML 모델 입력 준비)

#### 💡 상세 학술 메커니즘
이 데이터 파이프라인 아키텍처는 화학 물질 기반 QSAR(Quantitative Structure-Activity Relationship) 인공지능 모델 개발의 정석적인 **Data Wrangling** 아키텍처입니다. 
단순히 엑셀을 로드해 피처를 계산하는 1차원적인 구조에서 탈피하여, API 수집부터 물리적 구조 정화(Salts stripping), 라벨 노이즈 필터링(Label conflict removal), 위상적/구조적 중복 필터링을 다단계 파이프라인으로 설계함으로써 기계학습 학습 시 수렴을 극대화하고 이상치(Outlier)를 사전에 통제하는 완성도 높은 데이터 정화 파이프라인 시스템을 구현한 것입니다.